# L25 — Building the Clinic Input Model

**Module**: M07 | **Chapter**: 9 | **Lecture**: L25

## Learning Objectives
By the end of this notebook you will be able to:
1. Write a professional input model specification from fitted distributions.
2. Pass fitted distribution parameters into a SimPy model and verify they reproduce the empirical statistics.
3. Quantify model sensitivity to distributional choice via side-by-side simulation experiments.
4. Generate correlated random variates using the NORTA method.

---
> **Think → Trace → Code → Experiment → Interpret → Communicate**

This is the capstone lab for Module M07: we plug the fitted distributions from L22–L23
into a running SimPy clinic model and study how distributional assumptions affect performance.
---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import simpy
from scipy import stats
from dataclasses import dataclass, field
from typing import Callable

## 1. Fit Distributions to Both Datasets

In [ ]:
arr_data = pd.read_csv("../../../data/clinic_arrivals.csv")["interarrival_time"].to_numpy()
svc_data = pd.read_csv("../../../data/clinic_service_times.csv")["service_time"].to_numpy()

# Fit arrivals
_, scale_arr = stats.expon.fit(arr_data, floc=0)
a_arr, _, scale_arr_gam = stats.gamma.fit(arr_data, floc=0)

# Fit service times
s_svc, _, scale_svc_ln = stats.lognorm.fit(svc_data, floc=0)
a_svc, _, scale_svc_gam = stats.gamma.fit(svc_data, floc=0)

print("=== Fitted Input Distributions ===")
print(f"Arrivals — Exponential: mean = {scale_arr:.3f} min  (λ = {1/scale_arr:.4f}/min)")
print(f"Arrivals — Gamma:       shape={a_arr:.3f}, scale={scale_arr_gam:.3f},",
      f"mean={a_arr*scale_arr_gam:.3f}")
print()
print(f"Service  — Lognormal:   σ_log={s_svc:.3f}, scale={scale_svc_ln:.3f},",
      f"mean={stats.lognorm(s=s_svc, scale=scale_svc_ln).mean():.3f} min")
print(f"Service  — Gamma:       shape={a_svc:.3f}, scale={scale_svc_gam:.3f},",
      f"mean={a_svc*scale_svc_gam:.3f} min")

## 2. Minimal Clinic Simulation

A single-nurse triage queue: patients arrive, wait for the nurse, get triaged, depart.
We parameterise by `arrival_sampler` and `service_sampler` callables so we can
swap distributions without rewriting the simulation logic.

In [ ]:
@dataclass
class ClinicParams:
    arrival_sampler:  Callable   # rng → interarrival time
    service_sampler:  Callable   # rng → service time
    n_nurses:         int   = 1
    n_patients:       int   = 500
    label:            str   = 'Clinic'


def run_clinic(params: ClinicParams, seed: int) -> dict:
    rng  = np.random.default_rng(seed)
    env  = simpy.Environment()
    nurse = simpy.Resource(env, capacity=params.n_nurses)

    waits = []

    def patient(pid):
        arrival = env.now
        with nurse.request() as req:
            yield req
            wait = env.now - arrival
            waits.append(wait)
            svc = params.service_sampler(rng)
            yield env.timeout(svc)

    def arrivals():
        for pid in range(params.n_patients):
            env.process(patient(pid))
            ia = params.arrival_sampler(rng)
            yield env.timeout(ia)

    env.process(arrivals())
    env.run()

    return {
        'label':  params.label,
        'n':      len(waits),
        'Wq_mean': np.mean(waits),
        'Wq_std':  np.std(waits, ddof=1),
    }


print("Clinic model defined.")

## 3. Sensitivity to Distributional Choice

We run four combinations: {Exp, Gamma} × {Lognormal, Gamma} for arrivals × service.
30 replications each. The spread in $\hat{W}_q$ tells us how sensitive the output is
to which distribution we choose.

In [ ]:
N_REPS = 30
N_PATIENTS = 500

configurations = [
    ClinicParams(
        arrival_sampler=lambda rng: rng.exponential(scale_arr),
        service_sampler=lambda rng: rng.lognormal(mean=np.log(scale_svc_ln), sigma=s_svc),
        label='Exp-arrival / Lognormal-svc',
        n_patients=N_PATIENTS,
    ),
    ClinicParams(
        arrival_sampler=lambda rng: rng.exponential(scale_arr),
        service_sampler=lambda rng: rng.gamma(shape=a_svc, scale=scale_svc_gam),
        label='Exp-arrival / Gamma-svc',
        n_patients=N_PATIENTS,
    ),
    ClinicParams(
        arrival_sampler=lambda rng: rng.gamma(shape=a_arr, scale=scale_arr_gam),
        service_sampler=lambda rng: rng.lognormal(mean=np.log(scale_svc_ln), sigma=s_svc),
        label='Gamma-arrival / Lognormal-svc',
        n_patients=N_PATIENTS,
    ),
    ClinicParams(
        arrival_sampler=lambda rng: rng.gamma(shape=a_arr, scale=scale_arr_gam),
        service_sampler=lambda rng: rng.gamma(shape=a_svc, scale=scale_svc_gam),
        label='Gamma-arrival / Gamma-svc',
        n_patients=N_PATIENTS,
    ),
]

results = []
for cfg in configurations:
    reps = [run_clinic(cfg, seed=i)['Wq_mean'] for i in range(N_REPS)]
    mean_wq = np.mean(reps)
    ci_half = stats.t.ppf(0.975, df=N_REPS-1) * np.std(reps, ddof=1) / np.sqrt(N_REPS)
    results.append({'label': cfg.label, 'Wq_bar': mean_wq,
                    'ci_lo': mean_wq - ci_half, 'ci_hi': mean_wq + ci_half})
    print(f"{cfg.label:40s}  Wq={mean_wq:.2f} ± {ci_half:.2f} min")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

labels  = [r['label'] for r in results]
means   = [r['Wq_bar'] for r in results]
ci_lo   = [r['ci_lo'] for r in results]
ci_hi   = [r['ci_hi'] for r in results]

x = np.arange(len(labels))
ax.barh(x, means, xerr=[np.array(means)-ci_lo, np.array(ci_hi)-means],
        color='steelblue', alpha=0.7, capsize=5)
ax.set_yticks(x)
ax.set_yticklabels(labels, fontsize=9)
ax.set_xlabel('Mean wait in triage queue Wq (min)')
ax.set_title(f'Sensitivity to distributional choice  ({N_REPS} reps × {N_PATIENTS} patients)')
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Professional Input Model Specification

The deliverable from input modeling is not a plot — it is a written specification
that any analyst can read and implement.

In [ ]:
arr_mean = arr_data.mean()
arr_cv   = arr_data.std(ddof=1) / arr_mean
svc_mean = svc_data.mean()
svc_cv   = svc_data.std(ddof=1) / svc_mean

_, ks_p_arr = stats.kstest(arr_data, stats.expon(scale=scale_arr).cdf)
_, ks_p_svc = stats.kstest(svc_data, stats.lognorm(s=s_svc, scale=scale_svc_ln).cdf)

spec = f"""
INPUT MODEL SPECIFICATION — Primary Care Clinic
================================================

1. Patient Interarrival Times
   Distribution : Exponential(mean = {scale_arr:.2f} min)
   Sample size  : n = {len(arr_data)}
   Sample mean  : {arr_mean:.2f} min  |  CV = {arr_cv:.3f}
   Fitted mean  : {scale_arr:.2f} min
   KS p-value   : {ks_p_arr:.3f} (fail-to-reject at α=0.05)
   Rationale    : CV≈1 is characteristic of a Poisson process.
                  Exponential passes both KS and χ² tests.
   SimPy usage  : ia = rng.exponential({scale_arr:.2f})

2. Triage Nurse Service Times
   Distribution : Lognormal(σ_log = {s_svc:.3f}, scale = {scale_svc_ln:.3f})
   Sample size  : n = {len(svc_data)}
   Sample mean  : {svc_mean:.2f} min  |  CV = {svc_cv:.3f}
   Fitted mean  : {stats.lognorm(s=s_svc, scale=scale_svc_ln).mean():.2f} min
   KS p-value   : {ks_p_svc:.3f} (fail-to-reject at α=0.05)
   Rationale    : Right-skewed (positive skewness), CV<1 but >0.
                  Lognormal fits better than Exponential (lower AIC).
   SimPy usage  : svc = rng.lognormal(mean=np.log({scale_svc_ln:.3f}), sigma={s_svc:.3f})
"""
print(spec)

## 5. Correlated Arrivals and Service Times (NORTA Preview)

What if sicker patients both arrive in clusters *and* take longer to serve?
The NORTA (NORmal-To-Anything) method generates correlated non-Normal samples
by first sampling correlated Normals and then applying the inverse CDF.

In [ ]:
def norta_sample(dist_x, dist_y, rho, n, rng):
    """
    Generate n correlated pairs from dist_x and dist_y
    with linear correlation rho (applied in the Normal copula sense).
    """
    cov = [[1, rho], [rho, 1]]
    normals = rng.multivariate_normal([0, 0], cov, size=n)
    u = stats.norm.cdf(normals[:, 0])
    v = stats.norm.cdf(normals[:, 1])
    return dist_x.ppf(u), dist_y.ppf(v)


rng = np.random.default_rng(42)
arr_dist = stats.expon(scale=scale_arr)
svc_dist = stats.lognorm(s=s_svc, scale=scale_svc_ln)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, rho in zip(axes, [-0.5, 0.0, 0.5]):
    ia, svc = norta_sample(arr_dist, svc_dist, rho=rho, n=500, rng=rng)
    ia  = np.clip(ia,  0.01, None)   # ensure positive
    svc = np.clip(svc, 0.01, None)
    ax.scatter(ia, svc, s=5, alpha=0.4, color='steelblue')
    emp_r = np.corrcoef(ia, svc)[0, 1]
    ax.set_xlabel('Interarrival time (min)')
    ax.set_ylabel('Service time (min)')
    ax.set_title(f'ρ_target={rho}  ρ_empirical={emp_r:.2f}')
    ax.grid(True, alpha=0.3)

plt.suptitle('NORTA: correlated interarrival and service times', fontsize=11)
plt.tight_layout()
plt.show()

print("Note: negative correlation (ρ<0) means short interarrivals coincide with long service.")
print("This represents a busy-period effect: when patients arrive rapidly, each takes longer.")

---
## Try It Yourself

1. **Two-nurse clinic**: Change `n_nurses=2` in `ClinicParams` and re-run the sensitivity comparison. How much does doubling the nurses change Wq? Which distributional combination matters most with 2 nurses vs 1?

2. **Empirical distribution**: Instead of fitting a parametric family, pass the raw data directly as the sampler using `rng.choice(svc_data)`. How does the performance compare to the parametric fits? When would you prefer empirical over parametric?

3. **NORTA in the simulation**: Extend `run_clinic` to accept a joint sampler `joint_sampler(rng) → (ia, svc)` and use the NORTA pairs. Compare Wq at ρ=−0.5 vs ρ=0 vs ρ=+0.5. Explain intuitively why negative correlation reduces waiting.